# Training window comparison, walk-forward (1y / 2y / 3y, long and wide)

Same models and same comparison as `train_horizon_comparison_non_walk_forward.ipynb`, but instead of ONE 252-bar forecast period at the end of the data, the data is cut into consecutive **252-bar blocks** and every block is forecast by its own model, trained on the 1, 2 or 3 years right before it (walk-forward). Data: `4_ticker_data_2010_2026.csv`.

- Every block = one model, retrained when the block starts. The forecast days of one block become training days of the next one (rolling: the train window always keeps the last 252 / 504 / 756 bars).
- For each training length a **long** model (per-ticker forecasts + EWMA covariance bank) and a **wide** model (portfolio variance directly): six runs.
- The blocks are counted back from the end of the data, so the last block ends on the last day. The longer the training window, the fewer blocks fit in the data.
- Run-to-run comparisons (backtest, level/volatility, exceedance days) use the days **every** run has forecast (the blocks the longest-training run has); a separate table shows every block of every run.
- `RESIDUAL_SHARING = 'carry_over'`: when a later model's train window contains days an earlier model forecast, the earlier model's honest out-of-sample residual is used for those days instead of the new model's in-sample one. A cell compares it with `'none'`.

In [1]:
from itables import show
import numpy as np
import pandas as pd
import data_management as dm
import walk_forward as wf
from graph_helpers import (
    plot_var_vs_loss, plot_var_models_vs_loss, plot_models_output_vs_realized_variance, plot_level_and_volatility,
    var_vs_loss_date_picker, var_models_vs_loss_date_picker, plot_exceedance_days_focus,
)

In [2]:
TICKERS = ['AAPL', 'JPM', 'XOM', 'KO']

WEIGHTS = {
    'AAPL': 0.25,
    'JPM': 0.25,
    'XOM': 0.25,
    'KO': 0.25,
}

df = pd.read_csv(
    './data_general/4_ticker_data_2010_2026.csv',
    parse_dates=['Date'],
    index_col='Date',
)
print(df.index[0].date(), '->', df.index[-1].date(), '|', len(df), 'bars')

2010-01-04 -> 2026-06-30 | 4147 bars


In [3]:
FORECAST_WINDOW_BARS = 252                            # one model forecasts one block of 252 bars, then the window moves forward by 252 bars
TRAIN_BARS = {'1y': 252, '2y': 504, '3y': 756}        # training window lengths in bars
EXPANDING = False                                     # False: rolling train window (always TRAIN_BARS bars); True: grows from the first train bar
RESIDUAL_SHARING = 'carry_over'                       # 'none' or 'carry_over' (earlier models' out-of-sample residuals are reused)
WARMUP_BARS = 30                                      # bars before train_start used only to warm up rolling windows / EWMA

TARGET_SCALE = 10000                                  # the target (r^2) is multiplied by this for training, predictions are divided back
VAR_CONFIDENCE = 0.95                                 # 0.95 -> lower 5% quantile of the standardized residuals
VAR_WINDOW = 252                                      # VaR quantile window: the last N residuals
VAR_MIN_PERIODS = 30                                  # fewest prior residuals a day needs to get a VaR
EWMA_LAM = 0.94                                       # RiskMetrics decay: larger = forgets slower
EWMA_MIN_PERIODS = 30                                 # bars that seed the EWMA covariance

In [4]:
# Blocks are counted back from the end of the data: as many full FORECAST_WINDOW_BARS blocks as fit after
# (1 return-less first bar + warmup + train window). The price data is cut so the last block ends on the last day.
n = len(df)
calendars = {}
for name, train_bars in TRAIN_BARS.items():
    n_blocks = (n - WARMUP_BARS - 1 - train_bars) // FORECAST_WINDOW_BARS
    if n_blocks < 1:
        raise ValueError(f'{name}: not enough data for one {FORECAST_WINDOW_BARS}-bar block after {train_bars} train bars')
    first_bar = n - (WARMUP_BARS + 1 + train_bars + n_blocks * FORECAST_WINDOW_BARS)
    setting = wf.WalkForwardSetting(
        train_bars=train_bars, forecast_bars=FORECAST_WINDOW_BARS, expanding=EXPANDING,
        warmup_bars=WARMUP_BARS, residual_sharing=RESIDUAL_SHARING,
    )
    calendars[name] = setting.make_calendars(df.iloc[first_bar:], TICKERS, WEIGHTS)

pd.DataFrame({
    name: {
        'blocks': len(cals),
        'forecast days': sum(len(c.forecast_dates) for c in cals),
        'first train': f"{cals[0].train_data.index[0].date()} .. {cals[0].train_data.index[-1].date()}",
        'first block forecast': f"{cals[0].forecast_dates[0].date()} .. {cals[0].forecast_dates[-1].date()}",
        'last block forecast': f"{cals[-1].forecast_dates[0].date()} .. {cals[-1].forecast_dates[-1].date()}",
    }
    for name, cals in calendars.items()
})

[WARNING] train_start=2010-06-18 falls on Friday, expected Monday (using it as-is).
[WARNING] train_end=2011-06-16 falls on Thursday, expected Friday (using it as-is).
[WARNING] forecast_start=2011-06-17 falls on Friday, expected Monday (using it as-is).
[WARNING] [train] 8 weekday days had no data and were skipped (likely market holidays): 2010-07-05, 2010-09-06, 2010-11-25, 2010-12-24, 2011-01-17 and 3 more
[INFO] train set: 252 bars, 8 weekday days skipped
[WARNING] [forecast] 9 weekday days had no data and were skipped (likely market holidays): 2011-07-04, 2011-09-05, 2011-11-24, 2011-12-26, 2012-01-02 and 4 more
[INFO] forecast set: 252 bars, 9 weekday days skipped
[WARNING] train_start=2011-06-17 falls on Friday, expected Monday (using it as-is).
[WARNING] forecast_end=2013-06-19 falls on Wednesday, expected Friday (using it as-is).
[WARNING] [train] 9 weekday days had no data and were skipped (likely market holidays): 2011-07-04, 2011-09-05, 2011-11-24, 2011-12-26, 2012-01-02 an

,1y,2y,3y
blocks,15,14,13
forecast days,3780,3528,3276
first train,2010-06-18 .. 2011-06-16,2010-06-18 .. 2012-06-15,2010-06-18 .. 2013-06-19
first block forecast,2011-06-17 .. 2012-06-15,2012-06-18 .. 2013-06-19,2013-06-20 .. 2014-06-19
last block forecast,2025-06-30 .. 2026-06-30,2025-06-30 .. 2026-06-30,2025-06-30 .. 2026-06-30


In [5]:
# Long format: one model per block. Banks are SINGLE-USE and are created inside train_walk_forward for every block.
recipe_long = wf.ModelRecipe(
    'long', target_scale=TARGET_SCALE, mu_mode='constant', default_mu_hat=0.0,
    ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS,
)
fits_long = {name: wf.train_walk_forward(cals, recipe_long) for name, cals in calendars.items()}

[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2010-06-18, burn-in ends 2010-06-17, 0 date(s) dropped).
[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2011-06-17, burn-in ends 2011-06-16, 0 date(s) dropped).
[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2012-06-18, burn-in ends 2012-06-15, 0 date(s) dropped).


/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 2 value(s) <= 1e-12 at dates (first 5): [Timestamp('2011-10-27 00:00:00'), Timestamp('2011-11-02 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2011-08-09 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-

[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2013-06-20, burn-in ends 2013-06-19, 0 date(s) dropped).
[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2014-06-20, burn-in ends 2014-06-19, 0 date(s) dropped).
[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2015-06-22, burn-in ends 2015-06-19, 0 date(s) dropped).
[INFO] long train: 100

/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2020-03-02 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-standalone/core/EwmaVarianceBank.py:182: UserWarning: [EwmaVarianceBank.general_bulk_insert] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates [Timestamp('2020-03-03 00:00:00')] (value positions: [np.int64(2)]).
  sigma_diag_out[i] = DATA_QUALITY_check_variance_floor(S[i], dates=[dates[i]], context="EwmaVarianceBank.general_bulk_insert")


[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2019-06-24, burn-in ends 2019-06-21, 0 date(s) dropped).
[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2020-06-23, burn-in ends 2020-06-22, 0 date(s) dropped).
[INFO] long train: 1008 rows -> 1008 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 251/251 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2021-06-23, burn-in ends 2021-06-22, 0 date(s) dropped).
[INFO] long train: 100

/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 4 value(s) <= 1e-12 at dates (first 5): [Timestamp('2020-03-04 00:00:00'), Timestamp('2020-03-30 00:00:00'), Timestamp('2020-04-08 00:00:00'), Timestamp('2020-04-13 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 2 value(s) <= 1e-12 at dates (first 5): [Timestamp('2020-03-25 00:00:00'), Timestamp('2020-04-02 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_varia

[INFO] long train: 2016 rows -> 2016 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 503/503 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2019-06-24, burn-in ends 2019-06-21, 0 date(s) dropped).
[INFO] long train: 2016 rows -> 2016 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 503/503 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2020-06-23, burn-in ends 2020-06-22, 0 date(s) dropped).
[INFO] long train: 2016 rows -> 2016 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 1012 rows -> 1012 after dropna (0 dropped, 0.0%)
[INFO] train_dates_past_burn_in: 503/503 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2021-06-23, burn-in ends 2021-06-22, 0 date(s) dropped).
[INFO] long train: 201

In [6]:
# Wide format: one model per block predicts the PORTFOLIO variance directly (no per-ticker model, no EWMA bank).
recipe_wide = wf.ModelRecipe('wide', target_scale=TARGET_SCALE, mu_mode='constant', default_mu_hat=0.0)
fits_wide = {name: wf.train_walk_forward(cals, recipe_wide) for name, cals in calendars.items()}

[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)


/Users/kaan/Developer/MLProject-standalone/tree_models.py:84: UserWarning: [xgboost_train] DEGENERATE VARIANCE: 2 value(s) <= 1e-12 at dates (first 5): [Timestamp('2012-01-18 00:00:00'), Timestamp('2012-04-25 00:00:00')].
  predicted_variances = DATA_QUALITY_check_variance_floor(model.predict(X) / training_feature.target_scale, dates=common_index, context=f"{model_name}_train")
/Users/kaan/Developer/MLProject-standalone/core/ResidualBank.py:210: UserWarning: [general_bulk_insert] DEGENERATE VARIANCE: 2 value(s) <= 1e-12 at dates (first 5): [Timestamp('2012-01-18 00:00:00'), Timestamp('2012-04-25 00:00:00')].
  predicted_variances = DATA_QUALITY_check_variance_floor(predicted_variances, dates=dates,
/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2012-12-31 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.u

[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 252 bars -> 252 after dropna (0 drop

/Users/kaan/Developer/MLProject-standalone/tree_models.py:84: UserWarning: [xgboost_train] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2012-04-25 00:00:00')].
  predicted_variances = DATA_QUALITY_check_variance_floor(model.predict(X) / training_feature.target_scale, dates=common_index, context=f"{model_name}_train")
/Users/kaan/Developer/MLProject-standalone/core/ResidualBank.py:210: UserWarning: [general_bulk_insert] DEGENERATE VARIANCE: 1 value(s) <= 1e-12 at dates (first 5): [Timestamp('2012-04-25 00:00:00')].
  predicted_variances = DATA_QUALITY_check_variance_floor(predicted_variances, dates=dates,


[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 504 bars -> 504 after dropna (0 drop

/Users/kaan/Developer/MLProject-standalone/tree_models.py:38: UserWarning: [TreeModel.predict_variance_batch_wide with target_scale = 10000] DEGENERATE VARIANCE: 5 value(s) <= 1e-12 at dates (first 5): [Timestamp('2020-03-10 00:00:00'), Timestamp('2020-03-17 00:00:00'), Timestamp('2020-03-26 00:00:00'), Timestamp('2020-04-02 00:00:00'), Timestamp('2020-04-06 00:00:00')].
  variances = DATA_QUALITY_check_variance_floor(self.underlying_model.predict(rows) / self.target_scale, dates=feature_dates, context=f"TreeModel.predict_variance_batch_wide with target_scale = {forecast_feature.target_scale}" )
/Users/kaan/Developer/MLProject-standalone/core/ResidualBank.py:210: UserWarning: [general_bulk_insert] DEGENERATE VARIANCE: 5 value(s) <= 1e-12 at dates (first 5): [Timestamp('2020-03-11 00:00:00'), Timestamp('2020-03-18 00:00:00'), Timestamp('2020-03-27 00:00:00'), Timestamp('2020-04-03 00:00:00'), Timestamp('2020-04-07 00:00:00')].
  predicted_variances = DATA_QUALITY_check_variance_floor(pr

[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)
[INFO] wide train: 756 bars -> 756 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 253 bars -> 253 after dropna (0 dropped, 0.0%)


In [7]:
# The VaR of all blocks, one after the other = one out-of-sample var_df per run (all of the run's blocks).
var_setting = wf.VarSetting(var_confidence=VAR_CONFIDENCE, window=VAR_WINDOW, min_periods=VAR_MIN_PERIODS)

var_dfs = {name: wf.build_var_df(fits, var_setting, RESIDUAL_SHARING) for name, fits in fits_long.items()}
var_dfs_wide = {name: wf.build_var_df(fits, var_setting, RESIDUAL_SHARING) for name, fits in fits_wide.items()}

# distribution of the VaR level (varForecast) per run, over all of the run's blocks
pd.DataFrame({
    **{f'{name}_long': v['varForecast'] for name, v in var_dfs.items()},
    **{f'{name}_wide': v['varForecast'] for name, v in var_dfs_wide.items()},
}).describe()

,1y_long,2y_long,3y_long,1y_wide,2y_wide,3y_wide
count,3780.000000,3528.000000,3276.000000,3780.000000,3528.000000,3276.000000
mean,0.017864,0.016420,0.016165,0.019500,0.017828,0.018752
std,0.012675,0.010139,0.010337,0.013564,0.012859,0.017050
min,0.004002,0.004682,0.004948,0.000001,0.003807,0.000002
25%,0.010692,0.010571,0.010225,0.010655,0.010222,0.010614
50%,0.014415,0.013420,0.012938,0.015285,0.013515,0.013330
75%,0.020052,0.018779,0.018416,0.023066,0.020049,0.018579
max,0.119682,0.116901,0.128274,0.115090,0.111198,0.185739


In [8]:
# Every run in one place, keyed '<training length>_<model type>' (1y_long, 1y_wide, 2y_long, ...)
# residual_bank = the out-of-sample residual rows of all blocks (for the hover data and the variance graph)
runs = {}
for name in TRAIN_BARS:
    runs[f'{name}_long'] = {'var_df': var_dfs[name], 'residual_bank': wf.stitched_residual_bank(fits_long[name])}
    runs[f'{name}_wide'] = {'var_df': var_dfs_wide[name], 'residual_bank': wf.stitched_residual_bank(fits_wide[name])}

# The runs have different numbers of blocks, but they all end on the last day: the days EVERY run forecast are
# the last n_common ones. Run-to-run comparisons below use only those days (same days, same realized losses).
n_common = min(len(run['var_df']) for run in runs.values())
common_dates = df.index[-n_common:]
for run in runs.values():
    run['common_var_df'] = run['var_df'].loc[common_dates]

# The graphs and the calendar-based tables read one calendar: the common days as its forecast window and the
# bars before them as its train window. Only used for dates, realized variance and the train/forecast bands.
display_train_bars = min(max(TRAIN_BARS.values()), n - n_common - WARMUP_BARS - 1)
display_calendar = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start=df.index[n - n_common - display_train_bars], train_end=df.index[n - n_common - 1],
    forecast_start=df.index[n - n_common], forecast_end=df.index[-1],
    warmup_bars=WARMUP_BARS,
)
assert display_calendar.forecast_dates.equals(common_dates)
print(f'common period: {common_dates[0].date()} -> {common_dates[-1].date()} ({n_common} days)')
list(runs)

[WARNING] train_start=2010-06-18 falls on Friday, expected Monday (using it as-is).
[WARNING] train_end=2013-06-19 falls on Wednesday, expected Friday (using it as-is).
[WARNING] forecast_start=2013-06-20 falls on Thursday, expected Monday (using it as-is).
[WARNING] forecast_end=2026-06-30 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 28 weekday days had no data and were skipped (likely market holidays): 2010-07-05, 2010-09-06, 2010-11-25, 2010-12-24, 2011-01-17 and 23 more
[INFO] train set: 756 bars, 28 weekday days skipped
[WARNING] [forecast] 123 weekday days had no data and were skipped (likely market holidays): 2013-07-04, 2013-09-02, 2013-11-28, 2013-12-25, 2014-01-01 and 118 more
[INFO] forecast set: 3276 bars, 123 weekday days skipped
common period: 2013-06-20 -> 2026-06-30 (3276 days)


['1y_long', '1y_wide', '2y_long', '2y_wide', '3y_long', '3y_wide']

In [9]:
# Every block of every run, tested on its own (252 days per block, so the tests have some power).
block_rows = []
for key, run in runs.items():
    for block in wf.var_df_windows(run['var_df']):
        result = dm.run_full_backtest_suite_single_window(
            block['forecast_df'], block['window_start'], block['window_end'], VAR_CONFIDENCE, 0.95,
        )
        block_rows.append({
            'run': key, 'block': block['window_id'],
            'start': block['window_start'].date(), 'end': block['window_end'].date(),
            **{k: result[k] for k in (
                'n_days', 'n_exceedances', 'exceedance_rate', 'traffic_light_zone',
                'kupiec_pof_pass', 'christoffersen_ind_pass', 'christoffersen_cc_pass',
            )},
        })
pd.DataFrame(block_rows).set_index(['run', 'block'])

start         end  n_days  n_exceedances  exceedance_rate  \
run     block                                                                   
1y_long 0      2011-06-17  2012-06-15     252             22         0.087302   
        1      2012-06-18  2013-06-19     252              9         0.035714   
        2      2013-06-20  2014-06-19     252             10         0.039683   
        3      2014-06-20  2015-06-19     252             17         0.067460   
        4      2015-06-22  2016-06-20     252             13         0.051587   
...                   ...         ...     ...            ...              ...   
3y_wide 8      2021-06-23  2022-06-22     252             21         0.083333   
        9      2022-06-23  2023-06-23     252              7         0.027778   
        10     2023-06-26  2024-06-25     252              6         0.023810   
        11     2024-06-26  2025-06-27     252             19         0.075397   
        12     2025-06-30  2026-06-30     252             12         0.047619   

               traffic_light_zone  kupiec_pof_pass  christoffersen_ind_pass  \
run     block                                                                 
1y_long 0                       2            False                     True   
        1                       1             True                     True   
        2                       1             True                     True   
        3                       1             True                     True   
        4                       1             True                     True   
...                           ...              ...                      ...   
3y_wide 8                       2            False                     True   
        9                       1             True                     True   
        10                      1            False                     True   
        11                      2             True                     True   
        12                      1             True                     True   

               christoffersen_cc_pass  
run     block                          
1y_long 0                       False  
        1                        True  
        2                        True  
        3                        True  
        4                        True  
...                               ...  
3y_wide 8                        True  
        9                        True  
        10                       True  
        11                       True  
        12                       True  

[84 rows x 9 columns]

In [10]:
# Residual sharing 'none' vs 'carry_over' on the same trained models (no retraining, only the residuals the VaR reads differ).
# 'none': every model's VaR quantile reads its own in-sample residuals for the days it was trained on.
# The first block of a run has no earlier model to carry from, so it is the same in both rows.
rows = {}
for name in TRAIN_BARS:
    for model_type, fits in (('long', fits_long[name]), ('wide', fits_wide[name])):
        for sharing in ('none', 'carry_over'):
            v = wf.build_var_df(fits, var_setting, sharing)
            rows[(f'{name}_{model_type}', sharing)] = {
                'n_days': len(v),
                'n_exceedances': int((v['loss'] > v['varForecast']).sum()),
                'exceedance_rate': float((v['loss'] > v['varForecast']).mean()),
                'mean_varForecast': float(v['varForecast'].mean()),
            }
pd.DataFrame(rows).T.rename_axis(['run', 'residual_sharing'])

n_days  n_exceedances  exceedance_rate  \
run     residual_sharing                                           
1y_long none              3780.0          228.0         0.060317   
        carry_over        3780.0          202.0         0.053439   
1y_wide none              3780.0          251.0         0.066402   
        carry_over        3780.0          216.0         0.057143   
2y_long none              3528.0          208.0         0.058957   
        carry_over        3528.0          191.0         0.054138   
2y_wide none              3528.0          199.0         0.056406   
        carry_over        3528.0          196.0         0.055556   
3y_long none              3276.0          190.0         0.057998   
        carry_over        3276.0          179.0         0.054640   
3y_wide none              3276.0          193.0         0.058913   
        carry_over        3276.0          183.0         0.055861   

                          mean_varForecast  
run     residual_sharing                    
1y_long none                      0.015595  
        carry_over                0.017864  
1y_wide none                      0.016444  
        carry_over                0.019500  
2y_long none                      0.015229  
        carry_over                0.016420  
2y_wide none                      0.015678  
        carry_over                0.017828  
3y_long none                      0.015143  
        carry_over                0.016165  
3y_wide none                      0.015563  
        carry_over                0.018752

In [11]:
# From here on: the COMMON days only (the last n_common days, forecast by every run).
# Backtest suite, each run as one window. var_confidence must be the one the VaR was produced with.
backtest = pd.DataFrame({
    key: dm.backtest_forecast_period(display_calendar, run['common_var_df'], VAR_CONFIDENCE)
    for key, run in runs.items()
})
backtest

,1y_long,1y_wide,2y_long,2y_wide,3y_long,3y_wide
var_confidence,0.95,0.95,0.95,0.95,0.95,0.95
test_confidence,0.95,0.95,0.95,0.95,0.95,0.95
n_days,3276,3276,3276,3276,3276,3276
n_exceedances,171,189,178,189,179,183
exceedance_rate,0.052198,0.057692,0.054335,0.057692,0.05464,0.055861
expected_exceedance_rate,0.05,0.05,0.05,0.05,0.05,0.05
binomial_pass,True,False,True,False,True,True
traffic_light_zone,1,2,1,2,1,1
kupiec_pof_stat,0.328616,3.89672,1.261796,3.89672,1.443147,2.286128
kupiec_pof_critical_value,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459


In [12]:
# Summary: exceedances, whether the tests passed, and the mean VaR level
summary_rows = [
    'n_exceedances', 'exceedance_rate', 'expected_exceedance_rate', 'traffic_light_zone',
    'kupiec_pof_stat', 'kupiec_pof_critical_value', 'kupiec_pof_pass',
    'christoffersen_ind_pass', 'christoffersen_cc_pass',
]
summary = backtest.loc[summary_rows].copy()
summary.loc['mean_varForecast'] = [runs[key]['common_var_df']['varForecast'].astype(float).mean() for key in summary.columns]
summary

,1y_long,1y_wide,2y_long,2y_wide,3y_long,3y_wide
n_exceedances,171,189,178,189,179,183
exceedance_rate,0.052198,0.057692,0.054335,0.057692,0.05464,0.055861
expected_exceedance_rate,0.05,0.05,0.05,0.05,0.05,0.05
traffic_light_zone,1,2,1,2,1,1
kupiec_pof_stat,0.328616,3.89672,1.261796,3.89672,1.443147,2.286128
kupiec_pof_critical_value,3.841459,3.841459,3.841459,3.841459,3.841459,3.841459
kupiec_pof_pass,True,False,True,False,True,True
christoffersen_ind_pass,False,False,False,True,True,False
christoffersen_cc_pass,True,False,True,True,True,False
mean_varForecast,0.017537,0.018695,0.016518,0.017657,0.016165,0.018752


In [13]:
# Level and volatility suite (descriptive, no hypothesis tests): how HIGH is the VaR and how much does it MOVE.
# mean_tick_loss is the headline score (lower is better) = exceedance part (VaR too low) + overshoot part (VaR too high).
level_volatility = pd.DataFrame({
    key: dm.level_and_volatility_forecast_period(display_calendar, run['common_var_df'], VAR_CONFIDENCE)
    for key, run in runs.items()
})
level_volatility

,1y_long,1y_wide,2y_long,2y_wide,3y_long,3y_wide
var_confidence,0.950000,0.950000,0.950000,0.950000,0.950000,0.950000
n_days,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000,3276.000000
n_exceedances,171.000000,189.000000,178.000000,189.000000,179.000000,183.000000
exceedance_rate,0.052198,0.057692,0.054335,0.057692,0.054640,0.055861
expected_exceedance_rate,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
mean_tick_loss,0.001331,0.001401,0.001262,0.001350,0.001248,0.001432
tick_loss_exceedance_part,0.000399,0.000411,0.000382,0.000412,0.000386,0.000437
tick_loss_overshoot_part,0.000932,0.000990,0.000880,0.000939,0.000863,0.000995
mean_var,0.017537,0.018695,0.016518,0.017657,0.016165,0.018752
median_var,0.013369,0.014476,0.013157,0.013372,0.012938,0.013330


In [14]:
# Do the models miss on the same days? One row per day where at least one run has loss > VaR.
# n_models = how many runs exceeded that day (6 = all of them, 1 = only one run).
exceedance_days = dm.exceedance_days_by_model({key: run['common_var_df'] for key, run in runs.items()})
show(exceedance_days)

In [15]:
# How much do the runs agree? Number of days by how many runs missed them.
exceedance_days['n_models'].value_counts().sort_index().rename_axis('runs that missed the day').to_frame('days')

,days
runs that missed the day,
1,58
2,41
3,27
4,22
5,42
6,95


In [16]:
# Per (day, model): realized loss, VaR forecast and their difference (loss - VaR, positive = the VaR was exceeded by that much).
exceedance_details = dm.exceedance_day_details({key: run['common_var_df'] for key, run in runs.items()}, exceedance_days)
show(exceedance_details.reset_index())

In [17]:
# The exceedance days stacked in ONE figure: per day a graph (every run's VaR line + the realized loss) and,
# next to it, a table with loss, VaR and loss - VaR per run (exceeded rows in red).
# The common period is long now, so only the N_FOCUS_DAYS exceedance days with the largest realized loss are drawn.
N_FOCUS_DAYS = 15
focus_days = exceedance_days.sort_values('loss', ascending=False).head(N_FOCUS_DAYS).sort_index()
plot_exceedance_days_focus(
    {key: run['common_var_df'] for key, run in runs.items()},
    focus_days,
    window_bars=10,
)

In [18]:
# The VaR of every run (long and wide, 1y / 2y / 3y) on top of the same realized loss, common days
plot_var_models_vs_loss(
    {key: (run['common_var_df'], run['residual_bank']) for key, run in runs.items()},
    title='VaR Forecast vs Realized Loss (walk-forward, long / wide, 1y / 2y / 3y training)',
    calendar=display_calendar,
)

In [19]:
# Same graph with a date picker (needs ipywidgets): pick a start date and a window length, the graph jumps there.
var_models_vs_loss_date_picker(
    {key: (run['common_var_df'], run['residual_bank']) for key, run in runs.items()},
    title='VaR Forecast vs Realized Loss (walk-forward, long / wide, 1y / 2y / 3y training)',
    calendar=display_calendar,
)

In [20]:
# Model outputs (predicted_variance) against the EWMA realized variance, common days
plot_models_output_vs_realized_variance(
    display_calendar,
    {key: run['residual_bank'] for key, run in runs.items()},
    only_forecast_window=True,
)

In [21]:
# One graph per run over ALL of its blocks (hover shows predicted_variance, predicted_sigma, mu_hat and the standardized residual)
for key, run in runs.items():
    plot_var_vs_loss(run['var_df'], run['residual_bank'], title=f'VaR vs Realized Loss ({key}, walk-forward, all blocks)').show()